# Chapter 25: Two Pointers

The programs from Chapter 25 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Pair Sum in a Sorted List

`two_pointers.py`

In [1]:
"""Two pointers: two indexes that move through a sequence, usually from
both ends toward the middle or one chasing the other, so a pass that
would look at every pair looks at each item only once or twice."""


def pair_sum_sorted(nums, target):
    """Positions of two numbers in a sorted list adding up to target."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        total = nums[lo] + nums[hi]
        if total == target:
            return lo, hi
        if total < target:
            lo += 1  # need a bigger sum: only lo can give one
        else:
            hi -= 1  # need a smaller sum: only hi can give one
    return None


def dedupe_sorted(nums):
    """Remove duplicates from a sorted list in place; return the new
    length. `write` trails `read`, keeping each new value once."""
    write = 0
    for read in range(len(nums)):
        if read == 0 or nums[read] != nums[read - 1]:
            nums[write] = nums[read]
            write += 1
    del nums[write:]
    return write


def is_palindrome(text):
    """Ignoring case and anything that isn't a letter or digit."""
    lo, hi = 0, len(text) - 1
    while lo < hi:
        if not text[lo].isalnum():
            lo += 1
        elif not text[hi].isalnum():
            hi -= 1
        elif text[lo].lower() != text[hi].lower():
            return False
        else:
            lo, hi = lo + 1, hi - 1
    return True


def most_water(heights):
    """The most water held between two vertical lines. Moving the
    taller line inward can never help, so move the shorter one."""
    lo, hi, best = 0, len(heights) - 1, 0
    while lo < hi:
        width = hi - lo
        best = max(best, width * min(heights[lo], heights[hi]))
        if heights[lo] < heights[hi]:
            lo += 1
        else:
            hi -= 1
    return best


def three_sum(nums):
    """Every distinct triple adding up to 0: sort, fix the first item,
    then pair_sum on the rest. O(n^2) instead of O(n^3)."""
    nums, out = sorted(nums), []
    for i in range(len(nums) - 2):
        if i and nums[i] == nums[i - 1]:
            continue  # same first number: same triples again
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            total = nums[i] + nums[lo] + nums[hi]
            if total < 0:
                lo += 1
            elif total > 0:
                hi -= 1
            else:
                out.append((nums[i], nums[lo], nums[hi]))
                lo += 1
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1
    return out


def move_zeros(nums):
    """Move every 0 to the end in place, keeping the order of the rest."""
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]
            write += 1

## Demos and Tests

`two_pointers_demo.py`

In [2]:
"""The two-pointer solutions on small examples."""

from two_pointers import (
    dedupe_sorted,
    is_palindrome,
    most_water,
    move_zeros,
    pair_sum_sorted,
    three_sum,
)

print(pair_sum_sorted([1, 3, 4, 6, 8, 11], 10))
nums = [1, 1, 2, 3, 3, 3, 7]
print(dedupe_sorted(nums), nums)
print(is_palindrome("Was it a car or a cat I saw?"))
print(most_water([1, 8, 6, 2, 5, 4, 8, 3, 7]))
print(three_sum([-1, 0, 1, 2, -1, -4]))
nums = [0, 1, 0, 3, 12]
move_zeros(nums)
print(nums)

(2, 3)
4 [1, 2, 3, 7]
True
49
[(-1, -1, 2), (-1, 0, 1)]
[1, 3, 12, 0, 0]


## Timing

`bench_three_sum.py`

In [3]:
"""Three-sum: every triple (O(n^3)) against two pointers (O(n^2))."""

import itertools
import random

import bench
from two_pointers import three_sum

rng = random.Random(2026)


def brute(nums):
    return {
        tuple(sorted(c))
        for c in itertools.combinations(nums, 3)
        if sum(c) == 0
    }


def make(n):
    return ([rng.randint(-10_000, 10_000) for _ in range(n)],)


sizes = [100, 200, 400, 800]
results = {
    "all triples": bench.doubling(brute, make, sizes),
    "two pointers": bench.doubling(three_sum, make, sizes),
}
bench.table(results)

         n     all triples    two pointers
       100         11.1 ms          259 us
       200    91.2 ms x8.2    1.02 ms x3.9
       400     748 ms x8.2    4.14 ms x4.1
       800      6.2 s x8.3    16.8 ms x4.1
